In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
from src.utils import transition_ability_batched, update_v_history
import torch
from torch import nn
import torch.nn.functional as F

## 1. Configs

In [3]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 1000 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.04 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.06 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.2,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.1                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock
v_bar = 1.5


In [4]:
# Bounds of shock 
v_min = math.exp(-2 * sigma_v  / math.sqrt(1-rho_v**2))
v_max = math.exp( 2 * sigma_v  / math.sqrt(1-rho_v**2))

## 2. Helper functions and classes

In [5]:
def mean_across_agents(x): # Since agent number is fix, thus we can use mean instead of sum
    return torch.mean(x, dim=1, keepdim=True)


def calculate_price(a, v, h):
    a_aggregate, l_aggregate_effective = mean_across_agents(a), mean_across_agents(h*v)
    wage = A * (1-alpha) * ((a_aggregate/l_aggregate_effective) ** alpha)
    ret = A * alpha * (a_aggregate/l_aggregate_effective ** alpha)
    return wage, ret

def taxfunc(ibt, abt, taxparams=TAX_PARAMS):
    it = ibt - (1 - taxparams["tax_income"]) * (ibt**(1-taxparams["income_tax_elasticity"])/(1-taxparams["income_tax_elasticity"])) # individual after tax income
    at = abt - (1-taxparams["tax_saving"]/1-taxparams["saving_tax_elasticity"]) * (abt**(1-taxparams["saving_tax_elasticity"])) # individual after tax saving
    return it, at

def calculate_moneydisposable(wage, ret, v, h, a, delta, is_init=False):
    if is_init:
        ibt = wage * h * v   # individual before tax income
    else:
        ibt = wage * h * v + (1-delta+ret) * a   # individual before tax income

    it, at = taxfunc(ibt = ibt, abt=a)
    money_disposable = it + at

    return money_disposable, ibt


def output_transform(a, money_disposable):

    # The a here is saving rate coming from the NN output
    consumption = money_disposable * (1 - a)
    savings = money_disposable * a
    return consumption, savings


def laborfocloss(a, h, ibt, money_disposable, wage, v, taxparams=TAX_PARAMS):

    loss_foc =  -h ** (-gamma) + ((1-a)*money_disposable/(1+taxparams["tax_consumption"])) * \
        (wage * v) * (1 - taxparams["tax_income"]) * (ibt ** (-taxparams["income_tax_elasticity"]))
    
    return torch.abs(loss_foc)




In [6]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=3, dropout=0.1)

In [7]:
def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS):
    # 隨機產生初始資產與儲蓄
    moneydisposable = np.random.uniform(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    savings = np.random.uniform(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)

    # 第一種能力 shock (v1)
    v1 = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    v1 = v1 / np.mean(v1, axis=1, keepdims=True)

    # 第二種能力 shock (v2)
    v2 = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    v2 = v2 / np.mean(v2, axis=1, keepdims=True)

    # superstar 標誌 (v1 對應一組, v2 對應一組)
    is_superstar_v1 = np.zeros((required_batch_size, AGENTS), dtype=bool)
    is_superstar_v2 = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 轉為 tensor
    moneydisposable_t = torch.tensor(moneydisposable, dtype=torch.float32)
    savings_t = torch.tensor(savings, dtype=torch.float32)
    v1_t = torch.tensor(v1, dtype=torch.float32)
    v2_t = torch.tensor(v2, dtype=torch.float32)
    is_superstar_v1_t = torch.tensor(is_superstar_v1, dtype=torch.bool)
    is_superstar_v2_t = torch.tensor(is_superstar_v2, dtype=torch.bool)
    tax_params_t = tax_params

    # 回傳字典
    return {
        "moneydisposable": {"value": moneydisposable_t, "shape": tuple(moneydisposable_t.shape)},
        "savings": {"value": savings_t, "shape": tuple(savings_t.shape)},
        "v1": {"value": v1_t, "shape": tuple(v1_t.shape)},
        "v2": {"value": v2_t, "shape": tuple(v2_t.shape)},
        "is_superstar_v1": {"value": is_superstar_v1_t, "shape": tuple(is_superstar_v1_t.shape)},
        "is_superstar_v2": {"value": is_superstar_v2_t, "shape": tuple(is_superstar_v2_t.shape)},
        "tax_params": {"value": tax_params_t, "shape": tuple(tax_params_t.shape)},
    }


In [8]:
state = initial_state(required_batch_size=256)


In [9]:
def build_inputs(moneydisposable, savings, v, is_superstar, tax_params, carry_superstar=True):
    """
    回傳:
      features : (B, A, 2A + 2)          # 給模型輸入
      condi    : (B, A, Z)              # 稅制條件
      env_info : dict                   # 僅供環境轉移使用，不進模型
    """
    B, A = moneydisposable.shape

    # (B, Z) -> (B, A, Z)
    condi = tax_params.unsqueeze(1).expand(-1, A, -1)

    # (B, 2A) -> (B, A, 2A)
    sum_info = torch.cat([moneydisposable, v], dim=1)         # (B, 2A)
    sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)    # (B, A, 2A)

    # (B, A, 1) × 2
    money_self = moneydisposable.unsqueeze(-1)  # (B, A, 1)
    v_self     = v.unsqueeze(-1)                # (B, A, 1)

    # 給模型的 features
    features = torch.cat([sum_info_rep, money_self, v_self], dim=2)  # (B, A, 2A+2)

    # env_info: 包含不進模型的資訊
    env_info = {}

    # 儲蓄
    env_info["savings_self"] = savings.unsqueeze(-1)  # (B, A, 1)

    # 是否 super star
    superstar = None
    if carry_superstar and is_superstar is not None:
        if is_superstar.dim() == 0:                # scalar -> (B, A, 1)
            is_superstar = is_superstar.to(moneydisposable).view(1,1).expand(B, A).unsqueeze(-1)
        elif is_superstar.dim() == 2:              # (B, A) -> (B, A, 1)
            is_superstar = is_superstar.unsqueeze(-1)
        elif is_superstar.dim() == 3:              # (B, A, 1)
            pass
        else:
            raise ValueError("is_superstar must be scalar, (B,A), or (B,A,1)")
        superstar = is_superstar.to(features.dtype)
        env_info["superstar"] = superstar

    return features, condi, env_info


In [10]:
# resA = build_inputs(
#     moneydisposable=state["moneydisposable"]["value"],
#     savings=state["savings"]["value"],
#     v = state["v1"]["value"],
#     is_superstar = state["is_superstar_v1"]["value"],
#     tax_params=state["tax_params"]["value"]
# )

# resB = build_inputs(
#     moneydisposable=state["moneydisposable"]["value"],
#     savings=state["savings"]["value"],
#     v = state["v2"]["value"],
#     is_superstar = state["is_superstar_v2"]["value"],
#     tax_params=state["tax_params"]["value"]
# )

In [11]:
def process_agent(state, v_key, is_superstar_key, v_history):
    # 1️⃣ 構建輸入
    agent_state = build_inputs(
        moneydisposable=state["moneydisposable"]["value"],
        savings=state["savings"]["value"],
        v=state[v_key]["value"],
        is_superstar=state[is_superstar_key]["value"],
        tax_params=state["tax_params"]["value"]
    )

    # 2️⃣ Model forward
    acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
    out = model(agent_state[0], agent_state[1])
    at1, mut, ht = [acts[i](out[..., i]) for i in range(out.shape[-1])]
    at1, mut, ht = at1.squeeze(-1), mut.squeeze(-1), ht.squeeze(-1)

    # 3️⃣ Output transform
    consumption, savings = output_transform(at1, state["moneydisposable"]["value"])

    # 4️⃣ 計算價格
    wage, ret = calculate_price(savings, state[v_key]["value"], ht)

    # 5️⃣ 可支配收入
    money_disposable_t, ibt_t = calculate_moneydisposable(
        wage=wage, ret=ret, 
        v=state[v_key]["value"], h=ht, 
        a=state["savings"]["value"], delta=delta
    )

    # 6️⃣ 再次輸出轉換
    at1_transformed, savings = output_transform(a=at1, money_disposable=money_disposable_t)

    # 7️⃣ 能力轉移
    v_next, is_superstar_next = transition_ability_batched(
        v_prev=state[v_key]["value"],
        is_superstar_prev=state[is_superstar_key]["value"],
        v_history=v_history,
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

    # 8️⃣ 更新能力歷史
    v_history = update_v_history(v_history=v_history, v_next=v_next)

    return {
        "at1": at1,
        "mut": mut,
        "ht": ht,
        "consumption": consumption,
        "savings": savings,
        "wage": wage,
        "ret": ret,
        "money_disposable": money_disposable_t, # 為了match 前面state 的名稱
        "ibt_t": ibt_t,
        "v_next": v_next,
        "is_superstar_next": is_superstar_next,
        "v_history": v_history
    }


In [12]:
state.keys()

dict_keys(['moneydisposable', 'savings', 'v1', 'v2', 'is_superstar_v1', 'is_superstar_v2', 'tax_params'])

In [13]:
# def run_once(state):
#     v_history_A, v_history_B = None, None

#     result_A = process_agent(state, "v1", "is_superstar_v1", v_history_A)
#     result_B = process_agent(state, "v2", "is_superstar_v2", v_history_B)

#     pass
    # 你可以在這裡合併結果或回傳 dict
    # return {
    #     "A": result_A,
    #     "B": result_B
    # }


In [14]:
# ouo = run_once(state)

In [15]:
# ouo["A"]

In [ ]:
def run_once(state):
    v_history_A = None
    v_history_B = None


    state_A = build_inputs(
        moneydisposable=state["moneydisposable"]["value"],
        savings=state["savings"]["value"],
        v = state["v1"]["value"],
        is_superstar = state["is_superstar_v1"]["value"],
        tax_params=state["tax_params"]["value"]
    )

    state_B = build_inputs(
        moneydisposable=state["moneydisposable"]["value"],
        savings=state["savings"]["value"],
        v = state["v2"]["value"],
        is_superstar = state["is_superstar_v2"]["value"],
        tax_params=state["tax_params"]["value"]
    )


    # Handle Agnets action 
    acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
    outA = model(state_A[0], state_A[1])
    outB = model(state_B[0], state_B[1])
    
    at1_A, mut_A, ht_A = [acts[i](outA[..., i]) for i in range(outA.shape[-1])]
    at1_squeezed_A, mut_squeezed_A, ht_squeezed_A = at1_A.squeeze(-1), mut_A.squeeze(-1), ht_A.squeeze(-1)

    at1_B, mut_B, ht_B = [acts[i](outB[..., i]) for i in range(outB.shape[-1])]
    at1_squeezed_B, mut_squeezed_B, ht_squeezed_B = at1_B.squeeze(-1), mut_B.squeeze(-1), ht_B.squeeze(-1)

    # Output transform
    consumption_A, savings_A = output_transform(at1_squeezed_A, state["moneydisposable"]["value"])
    consumption_B, savings_B = output_transform(at1_squeezed_B, state["moneydisposable"]["value"])

    # Calculate price
    wage_A, ret_A = calculate_price(savings_A, state["v1"]["value"], ht_squeezed_A)
    wage_B, ret_B = calculate_price(savings_B, state["v2"]["value"], ht_squeezed_B)

    money_disposable_t_A, ibt_t_A = calculate_moneydisposable(wage = wage_A, ret = ret_A, 
                                               v = state["v1"]["value"], h = ht_squeezed_A, 
                                               a = state["savings"]["value"], delta = delta)
    money_disposable_t_B, ibt_t_B = calculate_moneydisposable(wage = wage_B, ret = ret_B, 
                                               v = state["v2"]["value"], h = ht_squeezed_B, 
                                               a = state["savings"]["value"], delta = delta)

    at1_transformed_A, savings_A = output_transform(a = at1_A, money_disposable = money_disposable_t_A)

    at1_transformed_B, savings_B = output_transform(a = at1_B, money_disposable = money_disposable_t_B)

    v_next_A, is_superstar_next_A = transition_ability_batched(
        v_prev=state["v1"]["value"],
        is_superstar_prev=state["is_superstar_v1"]["value"],
        v_history=v_history_A,            # 直接傳 tensor（或 None）
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )
    v_next_B, is_superstar_next_B = transition_ability_batched(
        v_prev=state["v2"]["value"],
        is_superstar_prev=state["is_superstar_v2"]["value"],
        v_history=v_history_B,            # 直接傳 tensor（或 None）
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )
    v_history_A = update_v_history(
        v_history=v_history_A,
        v_next=v_next_A
    )
    v_history_B = update_v_history(
        v_history=v_history_B,
        v_next=v_next_B
    )


In [17]:
# Handle Agnets action 
# at1 (sigmoid), mut(softplus), ht (sigmoid)
acts = [torch.sigmoid, lambda x: F.softplus(x) + 1e-6, torch.sigmoid]
outA = model(resA[0], resA[1])
outB = model(resB[0], resB[1])

at1, mut, ht = [acts[i](out[..., i]) for i in range(out.shape[-1])]
at1_squeezed, mut_squeezed, ht_squeezed = at1.squeeze(-1), mut.squeeze(-1), ht.squeeze(-1)

# Calculate env, and transform output 
wage, ret = calculate_price(a = state["savings"]["value"], v = state["v"]["value"], h = ht_squeezed)
money_disposable_t, ibt_t = calculate_moneydisposable(wage = wage, ret = ret, 
                                               v = state["v"]["value"], h = ht_squeezed, 
                                               a = state["savings"]["value"], delta = delta)

at1_transformed, savings = output_transform(a = at1, money_disposable = money_disposable_t)

# Shock transtion 
v_history = None

v_next, is_superstar_next = transition_ability_batched(
        v_prev=state["v"]["value"],
        is_superstar_prev=state["is_superstar"]["value"],
        v_history=v_history,            # 直接傳 tensor（或 None）
        rho_v=rho_v, sigma_v=sigma_v,
        p=p, q=q, v_bar=v_bar,
        v_min=v_min, v_max=v_max
    )

v_history = update_v_history(
    v_history=v_history,
    v_next=v_next
)


# state transition 
print(wage.shape, ret.shape, money_disposable_t.shape, ibt_t.shape)


# at1, mut = torch.sigmoid(at1), torch.exp(mut) # saving rate between 0 and 1 / ensure mu > 0
# print(res2[0].shape, res2[1].shape)
# print(model(res2[0], res2[1]).shape) # to log outputs
# print(flatten_last(model(res2[0], res2[1]))[0].shape) # to calculate loss

NameError: name 'resA' is not defined

In [ ]:
# 